# Day3_03B_CrewAI_Context_Chaining

## CrewAI - Tasks and Context Chain

**Duration:** 20 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Session:** Day 3 - CrewAI  

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand what **context chaining** means in CrewAI.
- Explain how one task output becomes input for another task.
- Create sequential tasks where later tasks depend on earlier outputs.
- Build a Researcher → Writer → Reviewer workflow.
- Relate context chaining to academic and enterprise workflows.

---

## Previous Notebook Recap

In the previous notebook, we built our first CrewAI workflow.

We learned four core concepts:

- **Agent** - who does the work
- **Task** - what needs to be done
- **Tool** - what the Agent can use
- **Crew** - how Agents collaborate

Now we will focus deeply on one very important idea:

> How does one Agent's output become useful to the next Agent?

That is called **context chaining**.


# 1. Why Context Chaining Matters

Imagine preparing an FDP module.

The workflow may look like this:

```text
Researcher researches topic
        ↓
Writer uses research to create module
        ↓
Reviewer uses both research and module to improve output
```

The Writer should not start from zero.

The Writer should use the Researcher's output.

The Reviewer should not review blindly.

The Reviewer should use the Writer's output and possibly the Researcher's output.

This passing of information between tasks is called **context chaining**.


# 2. Simple Teaching Analogy

Think of context chaining like academic teamwork.

```text
Research Notes
   ↓
Lecture Module
   ↓
Reviewed Module
```

Each stage improves the previous stage.

No one starts from scratch.

---

## Without Context Chain

Each Agent works independently.

Result may be disconnected.

## With Context Chain

Each Agent builds on previous work.

Result becomes coherent and structured.


# 3. CrewAI Architecture for This Notebook

We will build this flow:

```text
Research Task
   Output: Research Brief
        ↓
Write Task
   Input: Research Brief
   Output: Lecture Module
        ↓
Review Task
   Input: Lecture Module + Research Brief
   Output: Improved Module
```

This maps directly to the slide concept:

> context = automatically passes outputs to the next tasks


# 4. Setup

Make sure the following packages are installed:

```bash
pip install crewai crewai-tools python-dotenv
```

The `.env` file should be in the project root.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from crewai import Agent, Task, Crew, Process

# 5. Load Environment Variables

We will use the standard FDP root `.env` loading pattern.


In [ ]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

# 6. Define the Topic

We will use one running topic throughout this notebook.


In [ ]:
topic = "Agentic AI for Engineering Education"

print("Topic:", topic)

# 7. Create the Agents

We will create three Agents:

1. **Researcher** - Finds and organizes information.
2. **Writer** - Converts research into a teaching module.
3. **Reviewer** - Reviews and improves the teaching module.

Notice how each Agent has a role, goal, and backstory.


In [ ]:
researcher = Agent(
    role="AI Education Researcher",
    goal="Research practical and beginner-friendly points about AI topics for engineering faculty",
    backstory="""
    You are an AI education researcher who understands engineering colleges,
    faculty development programs, and classroom teaching needs. You focus on
    simple explanations, practical examples, and current relevance.
    """,
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Faculty Training Content Writer",
    goal="Convert research notes into simple, structured FDP teaching modules",
    backstory="""
    You are an experienced faculty training content writer. You create material
    that is easy for engineering professors to teach and easy for students to understand.
    You prefer simple language, examples, and classroom activities.
    """,
    verbose=True,
    allow_delegation=False,
)

reviewer = Agent(
    role="Academic Content Reviewer",
    goal="Review FDP teaching modules for clarity, flow, usefulness, and classroom readiness",
    backstory="""
    You are a senior academic reviewer. You improve teaching material by checking
    whether it is clear, complete, practical, and suitable for faculty with minimal coding background.
    """,
    verbose=True,
    allow_delegation=False,
)

# Pause & Reflect

Each Agent has a different responsibility.

This is better than asking one Agent to do everything.

Now we will define Tasks and connect them through context.


# 8. Task 1 - Research Task

The Researcher will prepare a brief.

This is the first task, so it does not need any previous context.


In [ ]:
research_task = Task(
    description=f"""
    Research the topic: {topic}

    Prepare a concise research brief covering:
    1. Simple definition
    2. Why the topic matters for engineering faculty
    3. One DSU-style classroom example
    4. One enterprise example
    5. Common misconceptions beginners may have
    """,
    expected_output="A concise research brief with five clearly labelled sections.",
    agent=researcher,
)

# 9. Task 2 - Write Task with Context

The Writer should use the output of the Research Task.

This is where context chaining starts.

Notice this line:

```python
context=[research_task]
```

It tells CrewAI:

> Use the research_task output as context for this writing task.


In [ ]:
write_task = Task(
    description="""
    Using the research brief from the previous task, create a 10-minute teaching module.

    The teaching module should include:
    1. Title
    2. Simple explanation
    3. Classroom analogy
    4. One mini activity
    5. Three key teaching points
    """,
    expected_output="A structured 10-minute teaching module for engineering faculty.",
    agent=writer,
    context=[research_task],
)

# 10. Task 3 - Review Task with Context

The Reviewer should review the Writer's output.

It can also consider the Researcher's output.

So we pass both tasks as context:

```python
context=[research_task, write_task]
```

This helps the Reviewer check whether the final module is faithful to the research and useful for teaching.


In [ ]:
review_task = Task(
    description="""
    Review the teaching module.

    Check:
    1. Is the explanation simple?
    2. Is there a classroom analogy?
    3. Is the mini activity practical?
    4. Are the teaching points useful?
    5. Is the flow suitable for a 10-minute classroom explanation?

    Provide an improved final version.
    """,
    expected_output="An improved classroom-ready teaching module.",
    agent=reviewer,
    context=[research_task, write_task],
)

# 11. Create the Crew

We will use a sequential process.

Sequential process means tasks run in order:

```text
Research → Write → Review
```

This is necessary because each task depends on previous outputs.


In [ ]:
context_chain_crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[research_task, write_task, review_task],
    process=Process.sequential,
    verbose=True,
)

# 12. Run the Crew

Now we run the workflow.

Observe how later tasks use previous outputs automatically.


In [ ]:
result = context_chain_crew.kickoff()

print(result)

# 13. What Did We Just Build?

We built a context-chained CrewAI workflow.

```text
Researcher Output
      ↓
Writer Uses Research
      ↓
Reviewer Uses Research + Writing
      ↓
Final Improved Module
```

This is different from independent Agents.

The output is more coherent because each task builds on the previous one.


# 14. Why Context Chain Is Powerful

Context chaining is useful when:

- Work happens in stages.
- Later tasks depend on earlier results.
- Quality improves through review.
- Outputs must be consistent.
- Teams need a shared understanding.

---

## Academic Examples

- Research notes → Paper draft → Reviewer comments
- Syllabus draft → Lesson plan → Faculty review
- Assignment design → Rubric → Evaluation checklist

## Enterprise Examples

- Customer issue → RCA draft → Manager review
- Security scan → Risk summary → Compliance review
- Product requirement → Design note → QA test cases


# 15. Modify the Workflow

Let us change the topic and run the same context chain again.

Try topics like:

- Prompt Engineering for Faculty
- RAG in Engineering Education
- AI Ethics in Classroom Teaching
- Multi-Agent Systems for Software Testing


In [ ]:
topic = "RAG in Engineering Education"

research_task.description = f"""
Research the topic: {topic}

Prepare a concise research brief covering:
1. Simple definition
2. Why the topic matters for engineering faculty
3. One DSU-style classroom example
4. One enterprise example
5. Common misconceptions beginners may have
"""

result = context_chain_crew.kickoff()

print(result)

# 16. Teaching Point: Context Is Not Memory

This is important.

Context chaining is not the same as memory.

| Context Chain | Memory |
|---|---|
| Passes task outputs within current workflow | Remembers across multiple runs |
| Temporary | Persistent |
| Used inside a Crew execution | Used across sessions |
| Great for workflows | Great for personalization |

In this notebook, we are using **context**, not long-term memory.


# 17. Classroom Exercise

Create a context chain for:

> Preparing a student assignment

Use three tasks:

1. Concept Task - explain the topic.
2. Assignment Task - create assignment questions.
3. Rubric Task - create evaluation rubric using the assignment.

Agents:

- Concept Explainer
- Assignment Designer
- Rubric Reviewer


In [ ]:
# Exercise Starter Code

concept_agent = Agent(
    role="Concept Explainer",
    goal="Explain technical topics simply for engineering students",
    backstory="You explain concepts clearly with examples.",
    verbose=True,
    allow_delegation=False,
)

assignment_agent = Agent(
    role="Assignment Designer",
    goal="Create meaningful student assignments",
    backstory="You design assignments that test understanding and application.",
    verbose=True,
    allow_delegation=False,
)

rubric_agent = Agent(
    role="Rubric Reviewer",
    goal="Create clear evaluation rubrics",
    backstory="You help faculty evaluate student submissions fairly.",
    verbose=True,
    allow_delegation=False,
)

# TODO:
# 1. Create concept_task
# 2. Create assignment_task with context=[concept_task]
# 3. Create rubric_task with context=[concept_task, assignment_task]
# 4. Create Crew
# 5. Run kickoff()


# 18. Challenge Exercise

Extend the workflow to four tasks:

```text
Research
   ↓
Write Module
   ↓
Create Quiz
   ↓
Review Everything
```

The final Review Task should use context from all previous tasks.

This will show how a Crew can build complete FDP material in stages.


# 19. Common Mistakes

Avoid these mistakes:

1. Forgetting to use `context=[previous_task]`.
2. Passing the wrong task as context.
3. Making tasks independent when they should be chained.
4. Writing vague expected outputs.
5. Creating too many context dependencies unnecessarily.
6. Assuming context chain is the same as memory.
7. Not checking whether later outputs actually used earlier information.


# 20. Key Takeaways

In this notebook, we learned:

- Context chaining passes task outputs to later tasks.
- `context=[task]` is how CrewAI connects task outputs.
- Sequential process is best when tasks depend on previous outputs.
- Research → Write → Review is a common and useful context chain.
- Context chaining creates more coherent and higher-quality outputs.
- Context is temporary within the workflow; memory is different.

---

## Final Mental Model

```text
Task 1 Output
   ↓
Task 2 uses Task 1 output
   ↓
Task 3 uses Task 1 + Task 2 output
   ↓
Final improved output
```

Context chaining is one of the most important ideas in CrewAI.
